# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a step-by-step workflow to load and explore the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata via Croissant
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print out the dataset's title and description
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

Let's retrieve and display the available record sets and their fields, referencing everything by their `@id` as per Croissant best practices.

In [ ]:
# List all record sets by `@id`
record_sets = list(dataset.record_sets)
if not record_sets:
    raise ValueError('No record sets available in the dataset.')

print(f"Number of record sets: {len(record_sets)}\n")
for idx, record_set in enumerate(record_sets):
    print(f"[{idx}] Record Set @id: {record_set['@id']}")
    print(f"    Name: {record_set.get('name', '(no name)')}")
    fields = record_set.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print("    Fields:")
    for f in fields:
        # A field can be dict (`@id`, ...) or plain string
        f_id = f.get('@id', f) if isinstance(f, dict) else f
        print(f"      - {f_id}")
    print('')

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract all data from all record sets using their @id
dataframes = {}
for record_set in record_sets:
    rs_id = record_set['@id']
    print(f"Loading records from Record Set: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"  Loaded {len(df)} records. Columns: {list(df.columns)}\n")
    else:
        print("  No records found in this record set.\n")

# Pick a record set to explore for further analysis
if not dataframes:
    raise ValueError("No dataframes loaded from available record sets.")
main_rs_id = list(dataframes.keys())[0]
print(f"\nMain record set chosen for analysis: {main_rs_id}")
print("Columns in main record set:")
print(dataframes[main_rs_id].columns.tolist())
dataframes[main_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. We'll demonstrate with one numeric field and perform filtering and normalization. We'll also group by a relevant field if one is found. All column and field references are by their `@id` as required.

In [ ]:
# Pick out a numeric field for demonstration.
import numpy as np

df = dataframes[main_rs_id]
# List possible numeric fields
numeric_fields = df.select_dtypes(include=[np.number]).columns
if numeric_fields.any():
    numeric_field_id = numeric_fields[0]
    print(f"Selected numeric field: {numeric_field_id}")
else:
    raise ValueError('No numeric fields found for analysis.')

# Demonstration threshold - set at the 25th percentile as an example
threshold = df[numeric_field_id].quantile(0.25)
filtered_df = df[df[numeric_field_id] > threshold].copy()
print(f"Filtered records in {main_rs_id} where {numeric_field_id} > {threshold:.2f} (25th percentile):")
display(filtered_df.head())

# Normalize numeric field
norm_col = f"{numeric_field_id}_normalized"
filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized field '{numeric_field_id}':")
print(filtered_df[[numeric_field_id, norm_col]].head())

# If there's a string/categorical column, group by it
cat_fields = df.select_dtypes(include=['object', 'category']).columns
group_field = None
# Avoid grouping over identifier or free text fields
for c in cat_fields:
    if c != numeric_field_id and not c.lower().startswith('id') and df[c].nunique() < len(df) and df[c].nunique() > 1:
        group_field = c
        break
if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().to_frame()
    print(f"\nGrouped mean of '{numeric_field_id}' by '{group_field}':")
    display(grouped_df.head())
else:
    print("No suitable group field found.")

## 5. Visualization
Visualize the distribution of the selected numeric field and compare across groupings if possible.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field_id].dropna(), kde=True)
plt.title(f"Distribution of '{numeric_field_id}'")
plt.xlabel(numeric_field_id)
plt.ylabel('Frequency')
plt.show()

# If grouping field exists, show boxplot
if group_field:
    plt.figure(figsize=(10,4))
    sns.boxplot(x=group_field, y=numeric_field_id, data=filtered_df)
    plt.title(f"Boxplot of '{numeric_field_id}' by '{group_field}'")
    plt.show()

## 6. Conclusion
- Successfully loaded and parsed the FAIR² colorectal cancer survivors dataset using `mlcroissant`.
- Inspected the structure using only Croissant `@id` values for reliable data reference and extraction.
- Performed exploratory statistics and data normalization on a key numeric variable, and visualized the results.
- This workflow can be extended for more in-depth analysis, modeling, or integrating additional clinical record sets.